### Tools

- Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.
- Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a chat model. The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.

In [ ]:
# !pip uninstall langchian langchain-openai langchain-community -y
# !pip install langchain langchain-openai langchain-community

In [ ]:
from langchain.tools import tool

### Basic Tool Definition
- The simplest way to create a tool is with the **@tool** decorator.
- By default, the function’s docstring becomes the tool’s description that helps the model understand when to use it
- Type hints are required as they define the tool’s input schema.
- The docstring should be informative and concise to help the model understand the tool’s purpose.
- By default, the tool name comes from the function name. Override it when you need something more descriptive
- Override the auto-generated tool description for clearer model guidance


In [ ]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

### Agents
- Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.
- **create_agent** provides a production-ready agent implementation.
- An LLM Agent runs tools in a loop to achieve a goal. An agent runs until a stop condition is met - i.e., when the model emits a final output or an iteration limit is reached.









In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

# Creating an agent with the defined tools
agent = create_agent(model, tools=[search, get_weather])

In [ ]:
agent.invoke("WHat is the day today")

- Tools give agents the ability to take actions. 
- Agents go beyond simple model-only tool binding by facilitating:
    - Multiple tool calls in sequence (triggered by a single prompt)
    - Parallel tool calls when appropriate
    - Dynamic tool selection based on previous results
    - Tool retry logic and error handling
    - State persistence across tool calls

### Example #1

- pip uninstall langchain langchain-openai langchain-community -y
- pip install "langchain>=0.1.0" langchain-openai langchain-community

##### Read the API Keys

In [ ]:
f = open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-8\key-vault\openai\api.key")
openai_apikey = f.read().strip()
f.close()

In [ ]:
f = open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-7\key-vault\openweather\api.key")
openweather_apikey = f.read().strip()
f.close()
openweather_apikey

In [ ]:
city = "Bangalore"
!curl "http://api.openweathermap.org/data/2.5/weather?q=${city}&appid=${openweather_apikey}&units=metric"

##### Define the calculator tools

In [ ]:
from langchain.tools import tool

@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

##### Define the weather tools

In [ ]:
import requests
from langchain.tools import tool

@tool("get_weather", description="Get weather information for a location.")
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    api_key = openweather_apikey
    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        weather_desc = data['weather'][0]['description']
        temp = data['main']['temp']
        return f"Weather in {location}: {weather_desc}, {temp}°C"
    else:
        return f"Error fetching weather data for {location}."
    

##### Build the agent

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [ ]:
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=openai_apikey)

# Tools list
tools = [calculator, get_weather]

# Agent
agent = create_agent(llm, tools=tools)

##### Invoke the agent

In [ ]:
# Invoke agent to check a calculation
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the weather in Bangalore"}
    ]
})
print(response)

In [ ]:
result = response["messages"][-1].content
print(result)

In [ ]:
# Invoke agent to check calculator
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 25 * 4 + 10?"}
    ]
})
print(response)

In [ ]:
result = response["messages"][-1].content
print(result)

##### Design a database tool

In [ ]:
# Set up a simple user database using SQLite
import sqlite3

conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id TEXT PRIMARY KEY,
    name TEXT,
    authenticated INTEGER
)
""")

conn.commit()
conn.close()

In [ ]:
# Add some seed data

def seed_data():
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()

    users = [
        ("ML001", "Raj", 1),
        ("ML002", "Ram", 0),
        ("ML003", "Sham", 1)
    ]

    cursor.executemany("INSERT OR IGNORE INTO users VALUES (?, ?, ?)", users)

    conn.commit()
    conn.close()

seed_data()

In [ ]:
# Define a tool to interact with the user database
# Define a tool to interact with the user database

from langchain.tools import tool
import sqlite3
import re

@tool
def user_db_tool(query: str) -> str:
    """
    Interact with user database.
    """
    print("[Query RXD]", query)
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()

    query_lower = query.lower()

    try:
        # ----------------------------
        # LIST AUTH USERS
        # ----------------------------
        if "authenticated users" in query_lower:
            cursor.execute("SELECT id, name FROM users WHERE authenticated=1")
            rows = cursor.fetchall()
            return "\n".join([f"{r[0]} - {r[1]}" for r in rows]) or "No authenticated users found"

        # ----------------------------
        # ADD USER
        # ----------------------------
        elif "insert" in query_lower:

            # ✅ FIXED REGEX
            name_match = re.search(r"name is (\w+)", query, re.IGNORECASE)
            id_match = re.search(r"id is (\w+)", query, re.IGNORECASE)

            print("[Match]", name_match, id_match)

            if name_match and id_match:
                name = name_match.group(1)
                user_id = id_match.group(1)

                cursor.execute(
                    "INSERT INTO users (id, name, authenticated) VALUES (?, ?, ?)",
                    (user_id, name, 1)
                )
                conn.commit()

                return f"User {name} added with ID {user_id}"

            else:
                return "Could not parse user details. Use format: 'name is X, id is Y'"

        # ----------------------------
        # LIST ALL USERS
        # ----------------------------
        elif "list all users" in query_lower:
            cursor.execute("SELECT id, name, authenticated FROM users")
            rows = cursor.fetchall()
            return "\n".join([str(r) for r in rows]) or "No users found"

        else:
            return "Unsupported query"

    except Exception as e:
        return f"Error: {str(e)}"

    finally:
        conn.close()

In [ ]:

# Create an agent with the new tool

tools = [calculator, get_weather, user_db_tool]

agent = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
# Test the agent with a query that requires using the user database tool
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "List all authenticated users"}
    ]
})

print(response["messages"][-1].content)

In [ ]:
# Add some data using a query that requires using the user database tool
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "insert name smitha id ML888 add this user"}
    ]
})

print(response["messages"][-1].content)

In [ ]:
# Invoke agent to list authenticated users again to see the updated list
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "List all users"}
    ]
})

print(response["messages"][-1].content)

### Notes

**User → LLM → Tool → LLM → Tool → Final Answer** this loop is not written manually 

##### Tool Routing

The LLM decides:

"Should I call calculator or user_db_tool?"

LangChain:

- Parses tool calls
- Executes the tool
- Feeds result back to LLM

Multi-step reasoning loop (ReAct) Thought → Action → Observation → Thought → ...

Handled automatically.

If your prompt/tools are weak:

Agent may:
- choose wrong tool
- hallucinate
- loop unnecessarily

👉 LangChain executes, it doesn’t guarantee intelligence.

Guardrails / validation

LangChain won’t stop:

- duplicate DB inserts
- malformed queries
- unsafe operations

👉 You must add validation logic.

Tool design

You still must design:

- Clean tool interfaces
- Good descriptions
- Structured inputs

Bad tool:

def tool(input: str):  # ❌ vague

Better:

def add_user(name: str, user_id: str):  # ✅ clear

##### Multi-agent orchestration (by default)

You mentioned earlier:

planner, retrieval, code, testing, deployment agents

👉 LangChain does NOT automatically build this system.

You must design:

Planner → Executor → Validator → Monitor

##### Memory strategy

LangChain gives memory tools, but doesn’t decide:

- what to store
- when to retrieve
- how long to persist